In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import pyreadstat

In [ ]:
wb = ['Europe & Central Asia - WB',
'Middle East & North Africa - WB',
'Sub-Saharan Africa - WB',
'East Asia & Pacific - WB',
'South Asia - WB',
'Latin America & Caribbean - WB',
'North America',]

In [ ]:
iran_provinces = ['Alborz',
'Ardebil',
'Bushehr',
'Chahar Mahaal and Bakhtiari',
'East Azarbayejan',
'Fars',
'Gilan',
'Golestan',
'Hamadan',
'Hormozgan',
'Ilam',
'Isfahan',
'Kerman',
'Kermanshah',
'Khorasan-e-Razavi',
'Khuzestan',
'Kohgiluyeh and Boyer-Ahmad',
'Kurdistan',
'Lorestan',
'Markazi',
'Mazandaran',
'North Khorasan',
'Qazvin',
'Qom',
'Semnan',
'Sistan and Baluchistan',
'South Khorasan',
'Tehran',
'West Azarbayejan',
'Yazd',
'Zanjan']

In [ ]:
sdi = ['High SDI', 
'High-middle SDI',
'Middle SDI',
'Low-middle SDI',
'Low SDI',#'Global']

In [ ]:
country = [
'Zambia',
'Uganda', 
'Lesotho',
'Namibia',
'Singapore', 
'Taiwan',
'Poland', 
'Luxembourg',
'Netherlands',
'Suriname',
'Belgium', 
'Norway',
'Sweden',
'Jamaica',
'Costa Rica',
'Guyana',
'Peru',
 'Canada',
'El Salvador',
'Ecuador',
'The Bahamas',
'Colombia',
'Guatemala',
'Honduras',
'Albania',
'Trinidad and Tobago',
'United Arab Emirates', 
'United States',
'United Kingdom',
'Bolivia', 
'Panama',
'Cuba',
'Argentina',
'Nicaragua', 
'The Gambia',
'Sierra Leone', 
'Senegal',
'Mali',
'Niger',
'Nigeria',
'Sao Tome and Principe', 
'Mauritania',
'American Samoa',
'Samoa',
'Belarus',  
 'Moldova',
 'Indonesia',
'Lithuania',
'Papua New Guinea',
'Kiribati', 
'Turkmenistan',
'South Korea',
'North Korea',
'Japan',
'Iran',
'Iraq',
'Kuwait',
'Algeria', 
'Turkey', 
'Armenia',
'Guam',
'Austria',
'Italy',
'Ireland',
'Iceland',
'Malta',
 'Spain',
'Germany',
'Israel', 
'Bhutan',
'Angola',
'Democratic Republic of the Congo', 
'Congo',
'Ethiopia',
'Kenya',
'Madagascar',
'Rwanda',
'Mozambique',
'Botswana', 
'Tanzania', 
'Seychelles',
'Malawi', 
'Mauritius',
'Eritrea', 
 'Burundi',
'Guinea-Bissau',
'Liberia',
'Ghana',
'Afghanistan',
'Nepal',
'Pakistan', 
'Comoros', 
'India',
'Equatorial Guinea', 
'Djibouti',
'Jordan',
'Benin',
'Cape Verde',
'Venezuela',
"Cote d'Ivoire",
'Cameroon',
'Chad',
'Vietnam', 
'Ukraine', 
'Russian Federation',
'Romania',
'Hungary',
'Croatia',
 'Mongolia',
'Sri Lanka',
 'Bulgaria',
'Philippines',
'Azerbaijan',
'Brunei',
'Serbia',
'Australia',
'China',
'Fiji',
'Virgin Islands, U.S.', 
'Bangladesh',
'Yemen',
'Somalia',
'Zimbabwe',
'Burkina Faso',
'Andorra', 
'Brazil',
'Saudi Arabia', 
'Egypt', 
 'Syria',
'Qatar',
'Czech Republic',
'Macedonia',
'Kyrgyzstan', 
 'Montenegro',
'Libya',
 'Myanmar',
'Cambodia',
 'Greenland',
'Chile',
'Federated States of Micronesia',
'Marshall Islands', 
'Vanuatu',
'Dominican Republic',
'Portugal',
'Timor-Leste', 
 'Thailand',
'Denmark',
 'Grenada',
'Solomon Islands', 
'Paraguay',
'France',
'Estonia',
'Belize', 
'Maldives',
'Gabon',
'South Africa',
'Saint Lucia',   
'Antigua and Barbuda',
'New Zealand',
'Malaysia',
'Latvia', 
'Kazakhstan',
'Togo',
 'Bermuda',
'Laos',
'Tonga',
'Bahrain',
'Morocco', 
'Bosnia and Herzegovina',
'Cyprus',
'Haiti',
 'Switzerland',
'Northern Mariana Islands',
'Finland',
'Lebanon',
 'Palestine', 
  'Barbados',
'Oman',
'Tunisia',
'Central African Republic',
'Guinea',
'Dominica', 
'Uzbekistan',
'Slovenia',
'Slovakia',
'Saint Vincent and the Grenadines',
'Tajikistan',
'South Sudan',
'Sudan', 
'Puerto Rico',
'Georgia', 
'Mexico', 
'Uruguay',
'Eswatini',
'Greece','Global'
]

In [ ]:
main_data_wide_Rate = pd.read_csv('../results/model_data.csv')

In [ ]:
main_data_wide_Rate.head()

In [ ]:
Rate_output_data = pd.DataFrame()
Rate_var_comp1 = []

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- 0) Config
# If you want to keep PERtoINC, use:
# FEATURES = ['MIR','YLLtoYLD','DALtoPER','PERtoINC']
FEATURES = ['MIR','YLLtoYLD','DALtoPER']

df = main_data_wide_Rate.copy()

# --- 1) TRAIN on age-standardized rows (pooled across all locations/years/sex)
train = df[df['age_name'] == 'Age-standardized'].dropna(subset=FEATURES).copy()
scaler = StandardScaler().fit(train[FEATURES].values)
Z_train = scaler.transform(train[FEATURES].values)

pca = PCA(n_components=1).fit(Z_train)
pc1_train = pca.transform(Z_train).ravel()

# Orient so higher = better (should be negatively correlated with MIR)
if np.corrcoef(pc1_train, train['MIR'].values)[0,1] > 0:
    sign = -1
    pc1_train *= sign
else:
    sign = 1

# Keep training extrema (for diagnostics only)
lo_train, hi_train = pc1_train.min(), pc1_train.max()
print(f"[Diag] TRAIN pc1 min/max: {lo_train:.4f}, {hi_train:.4f}")
print(f"[Diag] PC1 variance explained (train, %): {pca.explained_variance_ratio_[0]*100:.1f}")

# Train medians for light imputation when scoring (avoids dropping rows)
train_medians = train[FEATURES].median()

# --- 2) FUNCTION to score any subset WITHOUT refitting (produces RAW PC1 only)
def score_subset_raw_pc1(sub: pd.DataFrame) -> pd.DataFrame:
    out = sub.copy()
    X = out[FEATURES].copy().fillna(train_medians)  # impute with TRAIN stats
    pc1 = pca.transform(scaler.transform(X.values)).ravel() * sign
    out['pc1_raw'] = pc1
    return out

# --- 3) APPLY to every sex × age combo (transform only; no per-slice scaling)
parts = []
for s in df['sex_name'].unique():
    for a in df['age_name'].unique():
        block = df[(df['sex_name'] == s) & (df['age_name'] == a)].copy()
        scored = score_subset_raw_pc1(block)
        parts.append(scored)
        print(f"Scored sex={s}, age={a}")

Rate_output_data = pd.concat(parts, axis=0)

# --- 4) Define ONE global 0–100 mapping over the FULL reporting universe
lo_all = Rate_output_data['pc1_raw'].min()
hi_all = Rate_output_data['pc1_raw'].max()
Rate_output_data['pca_score'] = 100 * (Rate_output_data['pc1_raw'] - lo_all) / (hi_all - lo_all)

# Optional: clipped version for charts/tables (keeps analytics values intact)
Rate_output_data['pca_score_clipped'] = Rate_output_data['pca_score'].clip(0, 100)

print(f"[Diag] GLOBAL pc1 min/max used for 0–100: {lo_all:.4f}, {hi_all:.4f}")

In [ ]:
import pyreadstat
rate_var_comp1 = pd.DataFrame(
    [{'var_comp1': float(pca.explained_variance_ratio_[0] * 100),
      'model': 'global_age_standardized',
      'features': 'MIR,YLLtoYLD,DALtoPER'}]
)
pyreadstat.write_dta(rate_var_comp1, "variance_component_1.dta")

In [ ]:
# --- A) Loadings & contribution (report these)
FEATURES = ['MIR','YLLtoYLD','DALtoPER']  # matches the fit you just ran
loadings = pd.Series((pca.components_[0] * sign), index=FEATURES, name='loading')
contrib = ((loadings**2) / (loadings**2).sum()).rename('squared_loading_share')
load_tbl = pd.concat([loadings.round(3), contrib.round(3)], axis=1)
print(load_tbl)
print(f"PC1 variance explained: {pca.explained_variance_ratio_[0]*100:.1f}%")

# --- B) Correlation sanity checks on FULL scored data
chk = Rate_output_data.dropna(subset=['pca_score'] + FEATURES).copy()
corrs = chk[['pca_score'] + FEATURES].corr().loc['pca_score', FEATURES].sort_values()
print("Correlations of QCI with indicators:\n", corrs.round(3))
# Expect: negative with MIR, YLLtoYLD, DALtoPER.

# --- C) Range by sex × age (should now be 0–100 without out-of-range)
g = chk.groupby(['sex_name','age_name'])['pca_score']
print(g.agg(['count','min','median','max']).round(1))

# --- D) Presentation version (optional clipping; safe either way)
Rate_output_data['pca_score_clipped'] = Rate_output_data['pca_score'].clip(0, 100)

# --- E) Save outputs (reproducibility)
import joblib
artifacts = {
    'features': FEATURES,
    'scaler_mean_': scaler.mean_, 'scaler_scale_': scaler.scale_,
    'pca_components_': pca.components_, 'pca_explained_': pca.explained_variance_ratio_,
    'sign': sign, 'lo_all': lo_all, 'hi_all': hi_all,
    'train_lo': lo_train, 'train_hi': hi_train,
    'train_medians': train_medians.to_dict()
}
joblib.dump(artifacts, 'qci_pca_model.joblib')
Rate_output_data.to_csv('qci_scores_all_strata.csv', index=False)
print("Saved: qci_pca_model.joblib, qci_scores_all_strata.csv")

In [ ]:
ranked = Rate_output_data.query("sex_name=='Both' and age_name=='Age-standardized'")\
    .dropna(subset=['pca_score'])\
    .sort_values('pca_score')

print("Bottom 10:\n", ranked.head(10)[['location_name','year','pca_score']])
print("Top 10:\n", ranked.tail(10)[['location_name','year','pca_score']])

In [ ]:
Rate_output_data.head(2)

In [ ]:
wanted_columns = ['iso_location_name','location_name','haq_location_name','year','age_name','sex_name','pca_score']
qci = Rate_output_data.drop(
    columns=Rate_output_data.columns.difference(wanted_columns)
).copy()

In [ ]:
qci['qci'] = qci['pca_score']

In [ ]:
qci

In [ ]:
qci.to_csv('../results/qci.csv', index=False)

In [ ]:
Rate_output_data.to_csv('../results/qci_complete_data.csv')

In [ ]:
qci.iso_location_name.unique()